In [1]:
"""
Conformer Generation Pipeline - THREADED VERSION (Fork-safe) - WITH FLEXIBLE MOLECULE HANDLING
============================================================================================
"""

CONFIG = {
    "input":         "Mtb.xlsx",
    "sheet":         1,
    "name_col":      "Name",
    "smiles_col":    "SMILES",
    "outdir":        "conformers",

    "binding_site":  "Q-Loop",
    "ic50_col":      "IC50 μM",
    "ic50_max":      15,

    "energy_window": 5.0,
    "rmsd_cutoff":   0.7,
    "num_confs":     500,
    "max_iters":     200,
    "random_seed":   42,

    "force_field":      "MMFF94",
    "keep_hydrogens":   False,
    "max_conformers":   100,
    "verbose":          True,
    
    # Parallelization settings
    "n_workers":        4,              # Threads (safe for RDKit on all platforms)
    "use_threads":      True,           # Use threads instead of processes
    "molecule_timeout": 600,            # Timeout per molecule in seconds (increased for flexible molecules)
    "max_cluster_time": 120,            # Max time for RMSD clustering in seconds
    "max_opt_time":     300,            # Max time for optimization phase in seconds
}

import sys
import time
import logging
from pathlib import Path
from types import SimpleNamespace
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutureTimeoutError
import threading
from functools import wraps

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, Descriptors

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


# Thread-safe logger
class ThreadSafeLogger:
    def __init__(self, verbose=True):
        self.logger = logging.getLogger("confgen")
        self.lock = threading.Lock()
        
        if not self.logger.handlers:
            self.logger.setLevel(logging.DEBUG if verbose else logging.INFO)
            handler = logging.StreamHandler(sys.stdout)
            formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s','%H:%M:%S')
            handler.setFormatter(formatter)
            self.logger.addHandler(handler)
    
    def info(self, msg):
        with self.lock:
            self.logger.info(msg)
    
    def warning(self, msg):
        with self.lock:
            self.logger.warning(msg)
    
    def error(self, msg):
        with self.lock:
            self.logger.error(msg)
    
    def debug(self, msg):
        with self.lock:
            self.logger.debug(msg)


# ------------------------------------------------------------
# Problematic molecule detection and parameter adjustment
# ------------------------------------------------------------
def detect_problematic_molecule(mol, name, smiles, logger, prefix=""):
    """Detect problematic molecules and return adjusted parameters"""
    
    adjustments = {
        "num_confs": None,
        "max_iters": None,
        "rmsd_cutoff": None,
        "max_conformers": None,
        "reason": None
    }
    
    # Check for Aurachin D and similar polyenes
    aurachin_patterns = [
        "CC1=C(C(=O)C2=CC=CC=C2N1)C/C=C(\\C)/CC/C=C(\\C)/CCC=C(C)C",  # Aurachin D
        "CC1=C(C(=O)C2=CC=CC=C2N1)",  # Quinoline core pattern
        "CCC=C(C)C"  # Polyene tail pattern
    ]
    
    for pattern in aurachin_patterns:
        if pattern in smiles:
            adjustments["num_confs"] = 150
            adjustments["max_iters"] = 100
            adjustments["rmsd_cutoff"] = 1.2
            adjustments["max_conformers"] = 30
            adjustments["reason"] = "Flexible polyene detected (Aurachin-like)"
            logger.warning(f"{prefix}{name}: {adjustments['reason']} - Reducing parameters")
            return adjustments
    
    # Check rotatable bonds count
    num_rotatable = Descriptors.NumRotatableBonds(mol)
    if num_rotatable > 15:
        adjustments["num_confs"] = 200
        adjustments["max_iters"] = 120
        adjustments["rmsd_cutoff"] = 1.0
        adjustments["max_conformers"] = 40
        adjustments["reason"] = f"Very flexible molecule ({num_rotatable} rotatable bonds)"
        logger.warning(f"{prefix}{name}: {adjustments['reason']} - Reducing parameters")
    elif num_rotatable > 10:
        adjustments["num_confs"] = 300
        adjustments["max_iters"] = 150
        adjustments["rmsd_cutoff"] = 0.9
        adjustments["max_conformers"] = 60
        adjustments["reason"] = f"Flexible molecule ({num_rotatable} rotatable bonds)"
        logger.warning(f"{prefix}{name}: {adjustments['reason']} - Reducing parameters")
    
    # Check molecular weight
    mol_wt = Descriptors.ExactMolWt(mol)
    if mol_wt > 600:
        if adjustments["num_confs"] is None:
            adjustments["num_confs"] = 300
            adjustments["max_iters"] = 150
            adjustments["reason"] = f"High molecular weight ({mol_wt:.1f} Da)"
            logger.warning(f"{prefix}{name}: {adjustments['reason']} - Reducing parameters")
    
    return adjustments


# ------------------------------------------------------------
# Energy + optimization with timeout
# ------------------------------------------------------------
def optimize_conformer(mol, conf_id, force_field, max_iters):
    try:
        if force_field == "MMFF94s":
            props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant="MMFF94s")
            if props:
                ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=conf_id)
                if ff:
                    ff.Minimize(maxIts=max_iters)
                    return True, ff.CalcEnergy()

        # fallback UFF
        AllChem.UFFOptimizeMolecule(mol, confId=conf_id, maxIters=max_iters)
        ff = AllChem.UFFGetMoleculeForceField(mol, confId=conf_id)
        if ff:
            return True, ff.CalcEnergy()

    except Exception:
        pass

    return False, None


# ------------------------------------------------------------
# RMSD clustering with timeout
# ------------------------------------------------------------
def rmsd_clustering(mol, conf_ids, energies, rmsd_cutoff, max_keep, max_time=120, logger=None, name=""):
    """Energy-sorted greedy clustering with timeout"""
    start_time = time.time()
    
    pairs = sorted(zip(conf_ids, energies), key=lambda x: x[1])
    selected = [pairs[0][0]]
    
    for idx, (conf_id, energy) in enumerate(pairs[1:]):
        # Check timeout every 10 conformers
        if idx % 10 == 0:
            elapsed = time.time() - start_time
            if elapsed > max_time:
                if logger:
                    logger.warning(f"{name}: RMSD clustering timeout after {elapsed:.1f}s, stopping at {len(selected)} conformers")
                break
        
        keep = True
        for ref in selected:
            try:
                rmsd = rdMolAlign.GetBestRMS(mol, mol, conf_id, ref)
                if rmsd < rmsd_cutoff:
                    keep = False
                    break
            except Exception:
                continue

        if keep:
            selected.append(conf_id)
            if len(selected) >= max_keep:
                break
    
    elapsed = time.time() - start_time
    if logger:
        logger.debug(f"{name}: RMSD clustering completed in {elapsed:.1f}s, selected {len(selected)}/{len(pairs)} conformers")
    
    return selected


# ------------------------------------------------------------
# SDF Writer
# ------------------------------------------------------------
def write_sdf(mol, outdir, name, logger=None):
    """Write molecule with all conformers to SDF file."""
    safe = "".join(c if c.isalnum() else "_" for c in name)
    path = outdir / f"{safe}.sdf"

    try:
        conformers = mol.GetConformers()
        num_confs = len(conformers)

        if num_confs == 0:
            if logger:
                logger.warning(f"{name}: No conformers to write")
            return None

        mol.SetProp("_Name", name)
        mol.SetProp("Total_Conformers", str(num_confs))

        writer = Chem.SDWriter(str(path))
        writer.SetKekulize(False)

        for conf in conformers:
            conf_id = conf.GetId()

            if conf.HasProp("Energy"):
                mol.SetDoubleProp("Energy", conf.GetDoubleProp("Energy"))
                mol.SetProp("Energy_Unit", "RDKit_forcefield")
            if conf.HasProp("Rank"):
                mol.SetIntProp("Rank", conf.GetIntProp("Rank"))
            mol.SetProp("Conformer_Index", str(conf_id))

            writer.write(mol, confId=conf_id)

        writer.close()

        if logger:
            logger.debug(f"{name}: Wrote {num_confs} conformers to {path}")

        return path

    except Exception as e:
        if logger:
            logger.error(f"{name}: Failed to write SDF - {str(e)}")
        return None


# ------------------------------------------------------------
# Core function with all fixes
# ------------------------------------------------------------
def generate_conformers(smiles, name, cfg, logger, worker_id=None):
    """Generate conformers for a single molecule with robust error handling"""
    
    prefix = f"[W{worker_id}] " if worker_id is not None else ""
    mol_start_time = time.time()
    
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            logger.warning(f"{prefix}{name}: Invalid SMILES")
            return None, name, False, "Invalid SMILES"

        # Make a mutable copy of config for this molecule
        local_cfg = SimpleNamespace(**vars(cfg))
        
        # Detect problematic molecules and adjust parameters
        adjustments = detect_problematic_molecule(mol, name, smiles, logger, prefix)
        
        if adjustments["num_confs"] is not None:
            local_cfg.num_confs = adjustments["num_confs"]
        if adjustments["max_iters"] is not None:
            local_cfg.max_iters = adjustments["max_iters"]
        if adjustments["rmsd_cutoff"] is not None:
            local_cfg.rmsd_cutoff = adjustments["rmsd_cutoff"]
        if adjustments["max_conformers"] is not None:
            local_cfg.max_conformers = adjustments["max_conformers"]
        
        logger.info(f"{prefix}{name}: Processing with {local_cfg.num_confs} confs, {local_cfg.max_iters} iters, RMSD cutoff {local_cfg.rmsd_cutoff}")
        
        # Add hydrogens
        mol = Chem.AddHs(mol)
        
        # Calculate rotatable bonds for info
        num_rotatable = Descriptors.NumRotatableBonds(mol)
        logger.debug(f"{prefix}{name}: Molecule has {num_rotatable} rotatable bonds")

        # Embed conformers
        params = AllChem.ETKDGv3()
        params.randomSeed = local_cfg.random_seed if local_cfg.random_seed else (worker_id if worker_id else 42)
        params.numThreads = 1
        params.pruneRmsThresh = -1

        logger.debug(f"{prefix}{name}: Embedding {local_cfg.num_confs} conformers...")
        embed_start = time.time()
        conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=local_cfg.num_confs, params=params))
        embed_time = time.time() - embed_start
        
        logger.debug(f"{prefix}{name}: Embedding complete in {embed_time:.1f}s, got {len(conf_ids)} conformers")

        if not conf_ids:
            logger.warning(f"{prefix}{name}: No conformers embedded")
            return None, name, False, "No conformers embedded"

        # Optimize sequentially with timeout
        valid_conf_ids = []
        energies = []
        opt_start_time = time.time()
        max_opt_time = getattr(local_cfg, 'max_opt_time', 300)

        logger.debug(f"{prefix}{name}: Optimizing {len(conf_ids)} conformers (timeout: {max_opt_time}s)...")
        
        for i, conf_id in enumerate(conf_ids):
            # Check global optimization timeout
            if time.time() - opt_start_time > max_opt_time:
                logger.warning(f"{prefix}{name}: Optimization timeout reached after {i}/{len(conf_ids)} conformers")
                break
                
            success, energy = optimize_conformer(
                mol, conf_id, local_cfg.force_field, local_cfg.max_iters
            )

            if success and energy is not None:
                valid_conf_ids.append(conf_id)
                energies.append(energy)
                
            # Progress update
            if (i + 1) % 50 == 0:
                elapsed = time.time() - opt_start_time
                logger.debug(f"{prefix}{name}: Optimized {i+1}/{len(conf_ids)} conformers in {elapsed:.1f}s")

        if not energies:
            logger.warning(f"{prefix}{name}: No successful optimizations")
            return None, name, False, "No successful optimizations"

        energies = np.array(energies)
        e_min = energies.min()

        # Energy filter
        mask = energies <= e_min + local_cfg.energy_window
        filtered_conf_ids = [cid for cid, m in zip(valid_conf_ids, mask) if m]
        filtered_energies = [e for e, m in zip(energies, mask) if m]

        logger.debug(f"{prefix}{name}: Energy filter kept {len(filtered_conf_ids)}/{len(energies)} conformers (window={local_cfg.energy_window})")

        if not filtered_conf_ids:
            logger.warning(f"{prefix}{name}: No conformers in energy window")
            return None, name, False, "No conformers in energy window"

        # RMSD clustering with timeout
        max_cluster_time = getattr(local_cfg, 'max_cluster_time', 120)
        selected = rmsd_clustering(
            mol,
            filtered_conf_ids,
            filtered_energies,
            local_cfg.rmsd_cutoff,
            local_cfg.max_conformers,
            max_time=max_cluster_time,
            logger=logger,
            name=f"{prefix}{name}"
        )

        # Sort final by energy
        final_pairs = sorted(
            [(cid, energies[list(valid_conf_ids).index(cid)]) for cid in selected],
            key=lambda x: x[1]
        )

        logger.debug(f"{prefix}{name}: Selected {len(final_pairs)} conformers after clustering")

        # Build output molecule with selected conformers
        if not local_cfg.keep_hydrogens:
            mol_noH = Chem.RemoveHs(mol)
            out_mol = Chem.RWMol(mol_noH)
            out_mol.RemoveAllConformers()

            # Create atom index mapping (original -> no H)
            idx_map = {}
            j = 0
            for atom in mol.GetAtoms():
                if atom.GetAtomicNum() != 1:
                    idx_map[atom.GetIdx()] = j
                    j += 1

            for rank, (cid, energy) in enumerate(final_pairs):
                src_conf = mol.GetConformer(cid)
                new_conf = Chem.Conformer(out_mol.GetNumAtoms())

                for i_src, i_dst in idx_map.items():
                    pos = src_conf.GetAtomPosition(i_src)
                    new_conf.SetAtomPosition(i_dst, pos)

                new_conf.SetDoubleProp("Energy", float(energy))
                new_conf.SetIntProp("Rank", rank + 1)
                out_mol.AddConformer(new_conf, assignId=True)

            final_mol = out_mol.GetMol()

        else:
            out_mol = Chem.RWMol(mol)
            out_mol.RemoveAllConformers()

            for rank, (cid, energy) in enumerate(final_pairs):
                src_conf = mol.GetConformer(cid)
                new_conf = Chem.Conformer(out_mol.GetNumAtoms())

                for atom_idx in range(mol.GetNumAtoms()):
                    pos = src_conf.GetAtomPosition(atom_idx)
                    new_conf.SetAtomPosition(atom_idx, pos)

                new_conf.SetDoubleProp("Energy", float(energy))
                new_conf.SetIntProp("Rank", rank + 1)
                out_mol.AddConformer(new_conf, assignId=True)

            final_mol = out_mol.GetMol()

        # Set molecule properties
        final_mol.SetProp("_Name", name)
        final_mol.SetProp("SMILES", smiles)
        final_mol.SetIntProp("NumConformers", len(final_pairs))
        final_mol.SetProp("EnergyUnits", "RDKit_forcefield")
        
        # Add metadata about processing
        final_mol.SetProp("Processing_Time", f"{time.time() - mol_start_time:.1f}s")
        final_mol.SetProp("Original_Conformers", str(local_cfg.num_confs))

        # Write SDF file
        sdf_path = write_sdf(final_mol, Path(local_cfg.outdir), name, logger)
        
        if sdf_path is None:
            logger.warning(f"{prefix}{name}: Failed to write SDF file")
            return None, name, False, "Failed to write SDF"

        total_time = time.time() - mol_start_time
        logger.info(f"{prefix}{name}: ✓ Generated {len(final_pairs)} conformers in {total_time:.1f}s -> {sdf_path.name}")
        return final_mol, name, True, "Success"

    except Exception as e:
        total_time = time.time() - mol_start_time
        logger.error(f"{prefix}{name}: Exception after {total_time:.1f}s - {str(e)}")
        return None, name, False, str(e)


# ------------------------------------------------------------
# I/O
# ------------------------------------------------------------
def load_data(cfg, logger):
    df = pd.read_excel(cfg.input, sheet_name=cfg.sheet)

    if cfg.binding_site and "Binding site" in df.columns:
        df = df[df["Binding site"] == cfg.binding_site]

    if cfg.ic50_col and cfg.ic50_col in df.columns:
        df = df[df[cfg.ic50_col] < cfg.ic50_max]

    df = df.dropna(subset=[cfg.smiles_col, cfg.name_col])

    return list(zip(df[cfg.name_col], df[cfg.smiles_col]))


# ------------------------------------------------------------
# Parallel processing with robust timeout handling
# ------------------------------------------------------------
def process_molecules_parallel(records, cfg, logger):
    """Process molecules in parallel with aggressive timeout handling"""
    
    n_workers = cfg.n_workers if hasattr(cfg, 'n_workers') else 4
    molecule_timeout = cfg.molecule_timeout if hasattr(cfg, 'molecule_timeout') else 600
    
    logger.info(f"Starting parallel processing with {n_workers} workers (threads)")
    logger.info(f"Timeout per molecule: {molecule_timeout} seconds")
    
    successful = 0
    failed = 0
    results = []
    total = len(records)
    
    # Track running futures
    running_futures = {}
    
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        # Submit all jobs
        for idx, (name, smiles) in enumerate(records):
            worker_id = idx % n_workers
            future = executor.submit(generate_conformers, smiles, name, cfg, logger, worker_id)
            running_futures[future] = {"name": name, "start_time": time.time()}
        
        # Process completed futures with timeout checking
        completed = 0
        last_progress_time = time.time()
        
        while completed < total and running_futures:
            current_time = time.time()
            
            # Check for timeouts
            timed_out = []
            for future, info in list(running_futures.items()):
                elapsed = current_time - info["start_time"]
                if elapsed > molecule_timeout and not future.done():
                    logger.error(f"TIMEOUT: {info['name']} exceeded {molecule_timeout}s (elapsed: {elapsed:.1f}s)")
                    future.cancel()
                    timed_out.append(future)
                    failed += 1
                    results.append((info['name'], False))
                    completed += 1
                    logger.info(f"Progress: {completed}/{total} molecules processed ({completed*100//total}%)")
            
            # Remove timed out futures
            for future in timed_out:
                del running_futures[future]
            
            # Check for completed futures
            done_futures = [f for f in running_futures.keys() if f.done()]
            
            for future in done_futures:
                name = running_futures[future]["name"]
                try:
                    mol, mol_name, success, message = future.result(timeout=5)
                    if success and mol:
                        successful += 1
                    else:
                        failed += 1
                        logger.warning(f"Failed: {name} - {message}")
                    results.append((name, success))
                except FutureTimeoutError:
                    failed += 1
                    logger.error(f"Result timeout for {name}")
                    results.append((name, False))
                except Exception as e:
                    failed += 1
                    logger.error(f"Exception for {name}: {e}")
                    results.append((name, False))
                
                completed += 1
                logger.info(f"Progress: {completed}/{total} molecules processed ({completed*100//total}%)")
                del running_futures[future]
                last_progress_time = current_time
            
            # If nothing completed recently and we have running jobs, log status
            if current_time - last_progress_time > 30 and running_futures:
                active = [info["name"] for info in running_futures.values()]
                logger.info(f"Still waiting for {len(running_futures)} molecules: {', '.join(active[:3])}...")
                last_progress_time = current_time
            
            # Short sleep to prevent busy waiting
            if not done_futures and running_futures:
                time.sleep(5)
            
            # Safety check - if all remaining are stuck beyond timeout+30s, abort
            if running_futures and all(current_time - info["start_time"] > molecule_timeout + 30 
                                      for info in running_futures.values()):
                logger.error("All remaining workers appear stuck. Aborting remaining jobs.")
                for future in list(running_futures.keys()):
                    name = running_futures[future]["name"]
                    logger.error(f"Aborting stuck job: {name}")
                    future.cancel()
                    failed += 1
                    results.append((name, False))
                    completed += 1
                break
    
    logger.info(f"Parallel processing complete: {successful} succeeded, {failed} failed")
    return results


# ------------------------------------------------------------
# Sequential fallback (if parallel fails)
# ------------------------------------------------------------
def process_molecules_sequential(records, cfg, logger):
    """Fallback to sequential processing"""
    
    logger.info("Falling back to sequential processing...")
    
    successful = 0
    failed = 0
    total = len(records)
    
    for idx, (name, smiles) in enumerate(records):
        try:
            logger.info(f"Processing {idx+1}/{total}: {name}")
            mol, _, success, message = generate_conformers(smiles, name, cfg, logger, 0)
            if success and mol:
                successful += 1
            else:
                failed += 1
                logger.warning(f"Failed: {name} - {message}")
        except Exception as e:
            failed += 1
            logger.error(f"Exception for {name}: {e}")
        
        logger.info(f"Progress: {idx+1}/{total} molecules processed ({(idx+1)*100//total}%)")
    
    return successful, failed


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():
    cfg = SimpleNamespace(**CONFIG)
    logger = ThreadSafeLogger(cfg.verbose)

    outdir = Path(cfg.outdir)
    outdir.mkdir(exist_ok=True)

    # Load and filter data
    records = load_data(cfg, logger)
    logger.info(f"Loaded {len(records)} molecules")
    
    # Log any problematic patterns found
    problematic_count = 0
    for name, smiles in records:
        if "CC1=C(C(=O)C2=CC=CC=C2N1)" in smiles:
            problematic_count += 1
            logger.info(f"Detected Aurachin-like molecule: {name}")
    
    if problematic_count > 0:
        logger.info(f"Found {problematic_count} flexible polyene molecule(s) that will use reduced parameters")
    
    if not records:
        logger.warning("No molecules to process")
        return

    start_time = time.time()
    
    try:
        results = process_molecules_parallel(records, cfg, logger)
        successful = sum(1 for _, success in results if success)
        failed = len(results) - successful
    except Exception as e:
        logger.error(f"Parallel processing failed: {e}")
        successful, failed = process_molecules_sequential(records, cfg, logger)
    
    elapsed = time.time() - start_time
    
    logger.info("=" * 60)
    logger.info(f"FINAL SUMMARY:")
    logger.info(f"  Total molecules: {len(records)}")
    logger.info(f"  Successful: {successful}")
    logger.info(f"  Failed: {failed}")
    logger.info(f"  Time elapsed: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
    if len(records) > 0:
        logger.info(f"  Average time per molecule: {elapsed/len(records):.1f} seconds")
    logger.info("=" * 60)
    
    # List any failed molecules for easy reference
    if failed > 0:
        logger.info("Failed molecules:")
        for name, success in results:
            if not success:
                logger.info(f"  - {name}")


if __name__ == "__main__":
    main()

00:28:33 [INFO] Loaded 20 molecules
00:28:33 [INFO] Detected Aurachin-like molecule: Aurachin D
00:28:33 [INFO] Found 1 flexible polyene molecule(s) that will use reduced parameters
00:28:33 [INFO] Starting parallel processing with 4 workers (threads)
00:28:33 [INFO] Timeout per molecule: 600 seconds
00:28:33 [WARNING] [W0] Aurachin D: Flexible polyene detected (Aurachin-like) - Reducing parameters
00:28:33 [INFO] [W0] Aurachin D: Processing with 150 confs, 100 iters, RMSD cutoff 1.2
00:28:33 [INFO] [W2] CK-3-14: Processing with 500 confs, 200 iters, RMSD cutoff 0.7
00:28:33 [INFO] [W1] CK-3-22 (1T): Processing with 500 confs, 200 iters, RMSD cutoff 0.7
00:28:33 [INFO] [W3] RKA-307: Processing with 500 confs, 200 iters, RMSD cutoff 0.7
00:28:33 [DEBUG] [W0] Aurachin D: Molecule has 8 rotatable bonds
00:28:33 [DEBUG] [W2] CK-3-14: Molecule has 3 rotatable bonds
00:28:33 [DEBUG] [W1] CK-3-22 (1T): Molecule has 4 rotatable bonds
00:28:33 [DEBUG] [W3] RKA-307: Molecule has 2 rotatable bond